In [86]:
import pandas as pd

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score, log_loss, brier_score_loss

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

In [87]:
men_df = pd.read_csv("../data/m_tournament_training_dataset_advanced.csv")

In [88]:
def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5, print_results=True):
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        # convert scores to 0-1 range approximately
        y_prob = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        raise ValueError("Model does not support predict_proba or decision_function.")

    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    logloss = log_loss(y_test, y_prob)
    brier = brier_score_loss(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    results = {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall,
        "classification_report": report,
        "confusion_matrix": cm,
        "log_loss": logloss,
        "brier_score": brier,
        "auc": auc
    }

    if print_results:
        print("=== Classification Metrics ===")
        print(f"Accuracy:   {acc:.4f}")
        print(f"F1 Score:   {f1:.4f}")
        print(f"Precision:  {precision:.4f}")
        print(f"Recall:     {recall:.4f}")

        print("\n=== Classification Report ===")
        print(report)

        print("=== Confusion Matrix ===")
        print(cm)

        print("\n=== Probability Metrics ===")
        print(f"Log Loss:   {logloss:.4f}")
        print(f"Brier Score:{brier:.4f}")
        print(f"AUC:        {auc:.4f}")

    return results

In [89]:
model_results = []

def add_model_result(model_name, results):
    model_results.append({
        "Model": model_name,
        "Accuracy": results["accuracy"],
        "F1": results["f1_score"],
        "Precision": results["precision"],
        "Recall": results["recall"],
        "LogLoss": results["log_loss"],
        "BrierScore": results["brier_score"],
        "AUC": results["auc"]
    })

In [90]:
selected_cols = [
    "Season",
     "Target",
    "SeedNumDiff",
    "RankingDiff",
    "MarginDiff",
    "NetRatingDiff",
    "OffEffDiff",
    "DefEffDiff",
    "WinPctDiff",
    "OffDefGap",
    "DominanceScore",
    "NetRating_Margin_Interaction",
    "Margin_Ranking_Interaction",
    "TurnoverMarginDiff",
    "ReboundPctDiff"
]

In [91]:
men_df = men_df[keep_cols]

In [92]:
season_cutoff = 2023

train_df = men_df[men_df["Season"] < season_cutoff].copy()
test_df = men_df[men_df["Season"] >= season_cutoff].copy()

In [93]:
drop_cols = ["Season", "Team1ID", "Team2ID", "Target"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["Target"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["Target"]

In [94]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [95]:
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

In [96]:
random_forest_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [97]:
random_forest_results = evaluate_binary_classifier(random_forest_model, X_test, y_test)
add_model_result("Random Forest_keep", random_forest_results)

=== Classification Metrics ===
Accuracy:   0.7388
F1 Score:   0.7382
Precision:  0.7400
Recall:     0.7363

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.74      0.74      0.74       201
           1       0.74      0.74      0.74       201

    accuracy                           0.74       402
   macro avg       0.74      0.74      0.74       402
weighted avg       0.74      0.74      0.74       402

=== Confusion Matrix ===
[[149  52]
 [ 53 148]]

=== Probability Metrics ===
Log Loss:   0.5480
Brier Score:0.1853
AUC:        0.7971


In [98]:
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.07,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

In [99]:
xgb_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [100]:
xgb_results = evaluate_binary_classifier(xgb_model, X_test, y_test)
add_model_result("XGBoost_keep", xgb_results)

=== Classification Metrics ===
Accuracy:   0.6816
F1 Score:   0.6847
Precision:  0.6780
Recall:     0.6915

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.69      0.67      0.68       201
           1       0.68      0.69      0.68       201

    accuracy                           0.68       402
   macro avg       0.68      0.68      0.68       402
weighted avg       0.68      0.68      0.68       402

=== Confusion Matrix ===
[[135  66]
 [ 62 139]]

=== Probability Metrics ===
Log Loss:   0.5750
Brier Score:0.2000
AUC:        0.7675


In [101]:
voting_classifier = VotingClassifier(
    estimators=[
        ("xgb", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42
        )),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=5,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )),
        ("ada", AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.05,
            random_state=42
        ))
    ],
    voting="soft"
)

In [102]:
voting_classifier.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('xgb', ...), ('rf', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None


In [103]:
voting_results = evaluate_binary_classifier(voting_classifier, X_test, y_test)
add_model_result("Voting Classifier_keep", voting_results)

=== Classification Metrics ===
Accuracy:   0.7388
F1 Score:   0.7395
Precision:  0.7376
Recall:     0.7413

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.74      0.74      0.74       201
           1       0.74      0.74      0.74       201

    accuracy                           0.74       402
   macro avg       0.74      0.74      0.74       402
weighted avg       0.74      0.74      0.74       402

=== Confusion Matrix ===
[[148  53]
 [ 52 149]]

=== Probability Metrics ===
Log Loss:   0.5469
Brier Score:0.1852
AUC:        0.7947


In [104]:
results_df = pd.DataFrame(model_results)
results_df = results_df.sort_values(by=["LogLoss", "BrierScore", "AUC"], ascending=[True, True, False]).reset_index(drop=True)

In [105]:
results_df

,Model,Accuracy,F1,Precision,Recall,LogLoss,BrierScore,AUC
0,Voting Classifier_keep,0.738806,0.739454,0.737624,0.741294,0.546884,0.185216,0.794708
1,Random Forest_keep,0.738806,0.738155,0.740000,0.736318,0.548027,0.185266,0.797134
2,XGBoost_keep,0.681592,0.684729,0.678049,0.691542,0.574954,0.200010,0.767456


In [106]:
rf_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": random_forest_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nRandom Forest Feature Importances")
print(rf_importance_df.head(20))


Random Forest Feature Importances
                         Feature  Importance
0                    SeedNumDiff    0.277077
1                    RankingDiff    0.149958
7                      OffDefGap    0.115993
2                     MarginDiff    0.107540
3                  NetRatingDiff    0.102850
10    Margin_Ranking_Interaction    0.041743
4                     OffEffDiff    0.036881
6                     WinPctDiff    0.033514
9   NetRating_Margin_Interaction    0.033368
8                 DominanceScore    0.029919
5                     DefEffDiff    0.026178
11            TurnoverMarginDiff    0.023399
12                ReboundPctDiff    0.021580


In [107]:
xgb_importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nXGBoost Feature Importances")
print(xgb_importance_df.head(20))


XGBoost Feature Importances
                         Feature  Importance
0                    SeedNumDiff    0.321222
7                      OffDefGap    0.073179
3                  NetRatingDiff    0.069979
1                    RankingDiff    0.062992
10    Margin_Ranking_Interaction    0.059804
2                     MarginDiff    0.057925
9   NetRating_Margin_Interaction    0.055837
8                 DominanceScore    0.054307
11            TurnoverMarginDiff    0.054300
12                ReboundPctDiff    0.054289
5                     DefEffDiff    0.048586
6                     WinPctDiff    0.045453
4                     OffEffDiff    0.042127
